# Simple Insurance Pricing Demo

This is a simplified demonstration of the insurance pricing framework using causal inference.

## Overview
- Generate synthetic insurance data
- Analyze price-conversion relationships
- Optimize pricing for maximum profit
- Demonstrate causal inference concepts

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import our custom utilities
from utils import (
    generate_synthetic_insurance_data,
    calculate_profit_metrics,
    optimize_pricing_simple,
    plot_pricing_optimization,
    set_style
)

# Set up styling
set_style()

print("Libraries imported successfully!")
print("Ready to generate synthetic insurance data...")

## 1. Generate Synthetic Insurance Data

We'll create a dataset with realistic insurance pricing scenarios including:
- Customer demographics (age, income, region)
- Risk factors (risk score, previous claims)
- Market conditions (competitor pricing, volatility)
- Pricing treatment and conversion outcomes

In [ ]:
# Generate synthetic insurance dataset
print("Generating synthetic insurance dataset...")
df = generate_synthetic_insurance_data(n_samples=3000)

print(f"Dataset generated with {len(df)} customers")
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
df.head()

## 2. Basic Data Analysis

Let's examine the key characteristics of our synthetic dataset:

In [ ]:
# Basic statistics
print("Basic Dataset Statistics:")
print(f"Average age: {df['age'].mean():.1f} years")
print(f"Average income: ${df['income'].mean():,.0f}")
print(f"Average price: ${df['price'].mean():.2f}")
print(f"Conversion rate: {df['conversion'].mean():.2%}")
print(f"Average profit per customer: ${df['profit'].mean():.2f}")

# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Price distribution
axes[0, 0].hist(df['price'], bins=30, alpha=0.7, color='skyblue')
axes[0, 0].set_title('Price Distribution')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

# Conversion rate by price
price_bins = pd.cut(df['price'], bins=10)
conversion_by_price = df.groupby(price_bins)['conversion'].mean()
axes[0, 1].plot(range(len(conversion_by_price)), conversion_by_price.values, marker='o')
axes[0, 1].set_title('Conversion Rate by Price Level')
axes[0, 1].set_xlabel('Price Bin')
axes[0, 1].set_ylabel('Conversion Rate')

# Age distribution
axes[1, 0].hist(df['age'], bins=25, alpha=0.7, color='lightgreen')
axes[1, 0].set_title('Age Distribution')
axes[1, 0].set_xlabel('Age')
axes[1, 0].set_ylabel('Frequency')

# Profit by risk score
risk_bins = pd.cut(df['risk_score'], bins=10)
profit_by_risk = df.groupby(risk_bins)['profit'].mean()
axes[1, 1].plot(range(len(profit_by_risk)), profit_by_risk.values, marker='s', color='red')
axes[1, 1].set_title('Average Profit by Risk Level')
axes[1, 1].set_xlabel('Risk Bin')
axes[1, 1].set_ylabel('Average Profit ($)')

plt.tight_layout()
plt.show()

## 3. Profit & Loss Analysis

Let's analyze the financial performance of our current pricing strategy:

In [ ]:
# Calculate comprehensive P&L metrics
pnl_metrics = calculate_profit_metrics(df)

print("Profit & Loss Analysis:")
print("=" * 40)
print(f"Total customers: {pnl_metrics['total_customers']:,}")
print(f"Total conversions: {pnl_metrics['conversions']:,}")
print(f"Conversion rate: {pnl_metrics['conversion_rate']:.2%}")
print(f"Total revenue: ${pnl_metrics['total_revenue']:,.2f}")
print(f"Total profit: ${pnl_metrics['total_profit']:,.2f}")
print(f"Profit margin: {pnl_metrics['profit_margin']:.2%}")
print(f"Average CLV: ${pnl_metrics['average_clv']:,.2f}")

# Regional performance
print("\nRegional Performance:")
regional_metrics = df.groupby('region').agg({
    'conversion': 'mean',
    'price': 'mean',
    'profit': 'mean'
}).round(3)
print(regional_metrics)

## 4. Causal Inference: Price Effects

Let's analyze how price changes affect customer conversion using causal inference principles:

In [ ]:
# Analyze price-conversion relationship
print("Causal Analysis: Price Effects on Conversion")
print("=" * 50)

# Create price quartiles for treatment analysis
df['price_quartile'] = pd.qcut(df['price'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Calculate conversion rates by price quartile
conversion_by_quartile = df.groupby('price_quartile')['conversion'].agg(['mean', 'count'])
print("Conversion rates by price quartile:")
print(conversion_by_quartile)

# Treatment effect visualization
plt.figure(figsize=(12, 5))

# Box plot of conversion by price quartile
plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='price_quartile', y='conversion')
plt.title('Conversion Rate by Price Quartile')
plt.ylabel('Conversion (0/1)')

# Price sensitivity by customer segment
plt.subplot(1, 2, 2)
df['income_segment'] = pd.cut(df['income'], bins=3, labels=['Low', 'Medium', 'High'])
segment_conversion = df.groupby(['income_segment', 'price_quartile'])['conversion'].mean().unstack()
segment_conversion.plot(kind='bar', ax=plt.gca())
plt.title('Price Sensitivity by Income Segment')
plt.ylabel('Conversion Rate')
plt.xlabel('Income Segment')
plt.legend(title='Price Quartile')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

# Calculate treatment effects
q1_conversion = df[df['price_quartile'] == 'Q1']['conversion'].mean()
q4_conversion = df[df['price_quartile'] == 'Q4']['conversion'].mean()
treatment_effect = q4_conversion - q1_conversion

print(f"\nTreatment Effect Analysis:")
print(f"Q1 (lowest price) conversion rate: {q1_conversion:.2%}")
print(f"Q4 (highest price) conversion rate: {q4_conversion:.2%}")
print(f"Treatment effect (Q4-Q1): {treatment_effect:.2%}")

## 5. Pricing Optimization

Now let's find the optimal price that maximizes profit:

In [ ]:
# Optimize pricing for maximum profit
print("Pricing Optimization Analysis:")
print("=" * 40)

# Current performance
current_price = df['price'].mean()
current_profit = df['profit'].sum()

print(f"Current average price: ${current_price:.2f}")
print(f"Current total profit: ${current_profit:,.2f}")

# Run optimization
opt_results = optimize_pricing_simple(df, price_range=(200, 1500), n_points=100)

print(f"\nOptimization Results:")
print(f"Optimal price: ${opt_results['optimal_price']:.2f}")
print(f"Expected profit: ${opt_results['optimal_profit']:,.2f}")
print(f"Profit improvement: ${opt_results['optimal_profit'] - current_profit:,.2f}")
print(f"Improvement percentage: {((opt_results['optimal_profit'] - current_profit) / current_profit * 100):.1f}%")

# Plot optimization results
plot_pricing_optimization(opt_results)

## 6. Customer Segmentation Analysis

Let's analyze how different customer segments respond to pricing:

In [ ]:
# Customer segmentation analysis
print("Customer Segmentation Analysis:")
print("=" * 40)

# Risk-based segmentation
df['risk_category'] = pd.cut(df['risk_score'], bins=3, labels=['Low Risk', 'Medium Risk', 'High Risk'])

# Segment performance
segment_performance = df.groupby(['risk_category', 'region']).agg({
    'conversion': 'mean',
    'price': 'mean',
    'profit': 'mean'
}).round(3)

print("Performance by Risk Category and Region:")
print(segment_performance)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Conversion rate by segment
conversion_by_segment = df.groupby('risk_category')['conversion'].mean()
axes[0].bar(conversion_by_segment.index, conversion_by_segment.values, color='lightblue')
axes[0].set_title('Conversion Rate by Risk Category')
axes[0].set_ylabel('Conversion Rate')
axes[0].tick_params(axis='x', rotation=45)

# Price by segment
price_by_segment = df.groupby('risk_category')['price'].mean()
axes[1].bar(price_by_segment.index, price_by_segment.values, color='lightgreen')
axes[1].set_title('Average Price by Risk Category')
axes[1].set_ylabel('Average Price ($)')
axes[1].tick_params(axis='x', rotation=45)

# Profit by segment
profit_by_segment = df.groupby('risk_category')['profit'].mean()
axes[2].bar(profit_by_segment.index, profit_by_segment.values, color='lightcoral')
axes[2].set_title('Average Profit by Risk Category')
axes[2].set_ylabel('Average Profit ($)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Key Insights and Recommendations

Based on our analysis, let's summarize the key findings:

In [ ]:
# Summary insights
print("KEY INSIGHTS AND RECOMMENDATIONS")
print("=" * 50)

# Price sensitivity insights
price_correlation = df[['price', 'conversion']].corr().loc['price', 'conversion']
print(f"1. Price-Conversion Correlation: {price_correlation:.3f}")

# Segment insights
best_segment = segment_performance.groupby('risk_category')['profit'].mean().idxmax()
worst_segment = segment_performance.groupby('risk_category')['profit'].mean().idxmin()
print(f"2. Most Profitable Segment: {best_segment}")
print(f"3. Least Profitable Segment: {worst_segment}")

# Optimization insights
price_change = ((opt_results['optimal_price'] - current_price) / current_price) * 100
print(f"4. Recommended Price Change: {price_change:+.1f}%")

# Regional insights
best_region = regional_metrics['profit'].idxmax()
print(f"5. Most Profitable Region: {best_region}")

print("\nRECOMMENDATIONS:")
print("=" * 20)
print("• Implement dynamic pricing based on customer risk profiles")
print("• Focus marketing efforts on high-value segments")
print("• Consider regional pricing adjustments")
print("• Monitor price elasticity regularly")
print("• Test incremental pricing changes through A/B testing")

print("\nNEXT STEPS:")
print("=" * 15)
print("1. Explore the full 01_data_generation.ipynb notebook for advanced analysis")
print("2. Implement dynamic pricing strategies")
print("3. Set up A/B testing framework")
print("4. Integrate with production systems")
print("5. Monitor and iterate on pricing models")

## Conclusion

This demonstration shows how causal inference can be applied to insurance pricing problems. The framework includes:

1. **Data Generation**: Synthetic data with realistic causal relationships
2. **Causal Analysis**: Understanding price treatment effects
3. **Optimization**: Finding profit-maximizing prices
4. **Segmentation**: Targeted pricing strategies
5. **Validation**: Testing and measurement frameworks

This approach provides a solid foundation for implementing data-driven pricing strategies in insurance and other industries.